In [2]:
def shout(text):
    print(f"shout has been called with input {text}")
    return text.upper()


In [3]:
shout(text="hello")

shout has been called with input hello


'HELLO'

In [4]:
import gradio as gr
gr.interface(fn=shout,inputs="textbox",outputs="textboxs",flagging_mode="never").launch(inbrowse=True)

c:\Users\DELL\AIRLINE_CUSTOMER\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TypeError: 'module' object is not callable

In [ ]:
import gradio as gr


def shout(text):
    print(f"shout has been called with input {text}")
    return text.upper()


gr.Interface(
    fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never"
).launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


shout has been called with input hell


In [ ]:
system_message=""" 
you are helpful assistant for Airline called FlightAI.
Give short,courteous answer. 
Always accurate. If you don't know answer, say so. 
You help customers with:
1.Flight ticket prices
2.Flight status
3.Booking information
"""
from openai import OpenAI
def stream_ollama(prompt):
    messages=[
    {"role":"system","content":system_message},
    {"role":"user","content":prompt}
    ]
    stream=ollama.chat.completions.create(
        model="llama3.2",
        messages=messages,
        stream=True
    )
    result=""
    for chunk in stream:
        result+=chunk.choices[0].delta.content or ""
        yield result


In [ ]:
import gradio as gr
from openai import OpenAI
ollama=OpenAI(base_url="http://localhost:11434/v1",api_key="ollama")
message_input=gr.Textbox(label="your message",info="Enter a message for llama3.2",lines=7)
messages_output=gr.Markdown(label="Response:")
view=gr.Interface(
    fn=stream_ollama,
    title="OLLAMA",
    inputs=[message_input],
    outputs=[messages_output],
    examples=["Explain Airlines rules and regulations"],
    flagging_mode="never"

)
view.launch()

* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
load_dotenv(override=True)
openai_api_key=os.getenv("OPENAI_API_KEY")
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
MODEL="llama3.2"
if openai_api_key:
    print(f"Open API Key found")
else:
    print(f"Not found API Key")

Open API Key found


In [ ]:
system_message=""" 
you are helpful assistant for Airline called FlightAI.
Give short,courteous answer, no more than one sentence. 
Always accurate. If you don't know answer, say so. 
You help customers with:
1.Flight ticket prices
2.Flight status
3.Booking information
"""

In [ ]:
def chat(message,history):
    history=[{"role":h["role"],"content":h["content"]} for h in history]
    relevant_system_message=system_message
    messages=[{"role":"system","content":relevant_system_message}]+history+[{"role":"user","content":message}]
    stream=ollama.chat.completions.create(model=MODEL,messages=messages,stream=True)
    response="" 
    for chunk in stream:
      response +=chunk.choices[0].delta.content or ""
    yield response
    




In [ ]:
gr.ChatInterface(fn=chat,type="messages").launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "c:\Users\DELL\AIRLINE_CUSTOMER\.venv\Lib\site-packages\gradio\queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\DELL\AIRLINE_CUSTOMER\.venv\Lib\site-packages\gradio\route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\DELL\AIRLINE_CUSTOMER\.venv\Lib\site-packages\gradio\blocks.py", line 2191, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\DELL\AIRLINE_CUSTOMER\.venv\Lib\site-packages\gradio\blocks.py", line 1710, in call_function
    prediction = await utils.async_iteration(iterator)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\DELL\AIRLINE_CUSTOMER\.venv\Lib\site-packages\gradio\utils.py", line 760, in async_iteration
    return await ane

In [ ]:
ticket_prices={"london":"$799","paris":"$899","tokyo":"$1420","berlin":"$499"}

In [ ]:
def get_ticket_price(destination_city):
  print(f"Tool called for city: {destination_city}")
  # Convert input to lowercase so "London" and "london" both match
  city = destination_city.lower()
  price = ticket_prices.get(city, "unknown ticket price")
  return f"The price of a ticket to {destination_city} is {price}"

In [ ]:
# def get_ticket_price(destination_city):
#     print(f"Tool get_ticket_price called for {destination_city}")
#     city = destination_city.lower()
#     return ticket_prices.get(city, "Unknown")


price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city. Call this whenever you need to know the ticket price, for example when a customer asks 'How much is a ticket to this city'",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False,
    },
}


In [ ]:
#this included in alist of tools
tools=[{"type":"function","function":price_function}]
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': "Get the price of a return ticket to the destination city. Call this whenever you need to know the ticket price, for example when a customer asks 'How much is a ticket to this city'",
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

In [ ]:
def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    arguments = json.loads(tool_call.function.arguments)
    city = arguments.get("destination_city")
    price = get_ticket_price(city)
    response = {
        "role": "tool",
        "content": json.dumps({"destination_city": city, "price": price}),
        "tool_call_id": tool_call.id,
    }
    return response, city

In [ ]:
def chat(message, history):
    history=[{"role":h["role"],"content":h["content"]} for h in history]
    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )
    response = client.chat.completions.create(
        model=MODEL, messages=messages, tools=tools
    )

    if response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message
        tool_response, city = handle_tool_call(assistant_message)
        messages.append(assistant_message)
        messages.append(tool_response)
        messages.append({"role":"user","content":messages})
        response = client.chat.completions.create(
            model=MODEL, messages=messages,tools=tools
        )

    return response.choices[0].message.content

In [ ]:
# def handle_tool_call(message):
#     tool_call = message.tool_calls[0]
#     arguments = json.loads(tool_call.function.arguments)
#     city = arguments.get("destination_city")
#     price = get_ticket_price(city)
#     response = {
#         "role": "tool",
#         "content": json.dumps({"destination_city": city, "price": price}),
#         "tool_call_id": tool_call.id,
#     }
#     return response, city

In [ ]:
gr.ChatInterface(fn=chat,type="messages").launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7881
* To create a public link, set `share=True` in `launch()`.


In [ ]:
def chat(message, history):
    history=[{"role":h["role"],"content":h["content"]} for h in history]
    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )
    response = client.chat.completions.create(
        model=MODEL, messages=messages, tools=tools
    )

    while response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message
        tool_response, city = handle_tool_call(assistant_message)
        messages.append(assistant_message)
        messages.append(tool_response)
        messages.append({"role":"user","content":messages})
        response = client.chat.completions.create(
            model=MODEL, messages=messages,tools=tools
        )

    return response.choices[0].message.content

In [ ]:
import base64
from io import BytesIO
import json
import sqlite3
import gradio as gr
from openai import OpenAI
from PIL import Image

client = OpenAI()
MODEL = "llama3.2"
DB = "prices.db"

system_message = (
    "You are a helpful assistant for an Airline called FlightAI. "
    "Give short, courteous answers, no more than 1 sentence. "
    "Always be accurate. If you don't know the answer, say so."
)


# 1. Initialize SQLite database
def init_db():
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            "CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)"
        )
        sample_prices = [
            ("london", 799.0),
            ("paris", 899.0),
            ("tokyo", 1400.0),
            ("berlin", 499.0),
        ]
        cursor.executemany(
            "INSERT OR REPLACE INTO prices (city, price) VALUES (?, ?)",
            sample_prices,
        )
        conn.commit()


init_db()


# 2. Database lookup tool
def get_ticket_price(destination_city):
    print(f"DATABASE TOOL CALLED: Getting price for {destination_city}")
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            "SELECT price FROM prices WHERE city = ?",
            (destination_city.lower(),),
        )
        result = cursor.fetchone()
        return f"${result[0]}" if result else "No price data available for this city"


price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city. Call this whenever you need to know the ticket price, for example when a customer asks 'How much is a ticket to this city'",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False,
    },
}

tools = [{"type": "function", "function": price_function}]


# 3. Multimodal Output Helpers: Destination Image (DALL-E 3) & Voice (TTS)
def artist(city):
    image_response = client.images.generate(
        model="dall-e-3",
        prompt=f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style",
        size="1024x1024",
        n=1,
        response_format="b64_json",
    )
    image_data = base64.b64decode(image_response.data[0].b64_json)
    return Image.open(BytesIO(image_data))


def talker(message):
    response = client.audio.speech.create(
        model="llama3.2",
        voice="onyx",
        input=message,
    )
    # audio_path = "reply.mp3"
    # response.stream_to_file(audio_path)
    return response.text


# 4. Multimodal Input Helpers: Image Encoding & Audio Transcription
def encode_image(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def transcribe_audio(audio_path):
    with open(audio_path, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(
            model="whisper-1", file=audio_file
        )
    return transcript.text


# 5. Tool call processor
def handle_tool_calls(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get("destination_city")
            cities.append(city)
            price_details = get_ticket_price(city)
            responses.append(
                {
                    "role": "tool",
                    "content": price_details,
                    "tool_call_id": tool_call.id,
                }
            )
    return responses, cities


# 6. Process multimodal input (text + images + audio files) into Chatbot & OpenAI format
def add_user_input(user_msg, chat_history, api_messages):
    text = user_msg.get("text", "")
    files = user_msg.get("files", [])

    content_parts = []
    display_text = text

    for file_path in files:
        lower = file_path.lower()
        if lower.endswith((".png", ".jpg", ".jpeg", ".webp")):
            b64 = encode_image(file_path)
            content_parts.append(
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{b64}"},
                }
            )
            chat_history.append({"role": "user", "content": {"path": file_path}})
        elif lower.endswith((".mp3", ".wav", ".m4a", ".ogg")):
            spoken_text = transcribe_audio(file_path)
            display_text = f"{display_text} {spoken_text}".strip()

    if display_text:
        content_parts.insert(0, {"type": "text", "text": display_text})
        chat_history.append({"role": "user", "content": display_text})

    api_messages.append({"role": "user", "content": content_parts})
    return gr.MultimodalTextbox(value=None), chat_history, api_messages


# 7. Generate assistant reply, voice audio, and destination image
def bot_reply(chat_history, api_messages):
    messages = [{"role": "system", "content": system_message}] + api_messages
    response = client.chat.completions.create(
        model=MODEL, messages=messages, tools=tools
    )
    cities = []
    image = None

    while response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message
        tool_responses, called_cities = handle_tool_calls(assistant_message)
        cities.extend(called_cities)
        messages.append(assistant_message)
        messages.extend(tool_responses)
        messages.append({"role":"user","content":messages})
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools
        )

    reply = response.choices[0].message.content
    chat_history.append({"role": "assistant", "content": reply})
    api_messages.append({"role": "assistant", "content": reply})

    voice = talker(reply)
    if cities:
        image = artist(cities[0])

    return chat_history, api_messages, voice, image


# 8. Gradio Blocks Layout
with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_path = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_outpath = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

    message.submit(
        put_message_in_chatbot,  # pyright: ignore[reportUndefinedVariable]
        inputs=[message, chatbot],
        outputs=[message, chatbot],
    ).then(
        chat,
        inputs=chatbot,
        outputs=[chatbot, audio_outpath, image_path],
    )
ui.launch(inbrowser=True)





C:\Users\DELL\AppData\Local\Temp\ipykernel_21900\4218502377.py:198: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=500, type="messages")


* Running on local URL:  http://127.0.0.1:7894
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "c:\Users\DELL\AIRLINE_CUSTOMER\.venv\Lib\site-packages\gradio\queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\DELL\AIRLINE_CUSTOMER\.venv\Lib\site-packages\gradio\route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\DELL\AIRLINE_CUSTOMER\.venv\Lib\site-packages\gradio\blocks.py", line 2191, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\DELL\AIRLINE_CUSTOMER\.venv\Lib\site-packages\gradio\blocks.py", line 1698, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\DELL\AIRLINE_CUSTOMER\.venv\Lib\site-packages\anyio\to_thread.py", line 65, in run_sync
    r